## Cell 1 — Install Ultralytics (YOLOv11)

In [ ]:

# Run inside your existing `yolo11` conda env:
#   conda activate yolo11
#   pip install ultralytics
!pip install -q ultralytics


## Cell 2 — Paths and settings

In [ ]:

import os

BASE_DIR   = r"C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone"
IMAGES_DIR = os.path.join(BASE_DIR, "img")
ANNOTS_DIR = os.path.join(BASE_DIR, "xml")

YOLO_DATASET_DIR = os.path.join(BASE_DIR, "yolo_dataset")
YOLO_IMAGES_TRAIN = os.path.join(YOLO_DATASET_DIR, "images", "train")
YOLO_IMAGES_VAL   = os.path.join(YOLO_DATASET_DIR, "images", "val")
YOLO_LABELS_TRAIN = os.path.join(YOLO_DATASET_DIR, "labels", "train")
YOLO_LABELS_VAL   = os.path.join(YOLO_DATASET_DIR, "labels", "val")

for d in [YOLO_IMAGES_TRAIN, YOLO_IMAGES_VAL, YOLO_LABELS_TRAIN, YOLO_LABELS_VAL]:
    os.makedirs(d, exist_ok=True)

# Class setup. Your XMLs use <name>UAV</name>, so class 0 = UAV.
CLASS_NAMES = ["UAV"]

# Train/val split ratio (80/20 by default)
VAL_RATIO = 0.20

print("Dataset will be built at:", YOLO_DATASET_DIR)
print("Classes:", CLASS_NAMES)


Dataset will be built at: C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone\yolo_dataset
Classes: ['UAV']


## Cell 3 — Convert Pascal VOC XML -> YOLO .txt format

In [ ]:

from lxml import etree

def voc_to_yolo_line(xml_path, class_names):
    # Returns a list of YOLO-format label lines (class x_center y_center w h, normalized)
    tree = etree.parse(xml_path)
    root = tree.getroot()

    size = root.find("size")
    img_w = float(size.find("width").text)
    img_h = float(size.find("height").text)

    lines = []
    for obj in root.findall("object"):
        name = obj.find("name").text.strip()
        if name not in class_names:
            print(f"WARNING: unknown class '{name}' in {xml_path}, skipping this object")
            continue
        class_id = class_names.index(name)

        bnd = obj.find("bndbox")
        xmin = float(bnd.find("xmin").text)
        ymin = float(bnd.find("ymin").text)
        xmax = float(bnd.find("xmax").text)
        ymax = float(bnd.find("ymax").text)

        x_center = ((xmin + xmax) / 2) / img_w
        y_center = ((ymin + ymax) / 2) / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h

        lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")

    return lines

# Quick test on one file
test_xml = os.path.join(ANNOTS_DIR, os.listdir(ANNOTS_DIR)[0])
print("Testing on:", test_xml)
print(voc_to_yolo_line(test_xml, CLASS_NAMES))


Testing on: C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone\xml\00094.xml
['0 0.282031 0.361111 0.153125 0.169444']


## Cell 4 — Build the full YOLO dataset (train/val split + label conversion)

In [ ]:

import random
import shutil

image_files = sorted([f for f in os.listdir(IMAGES_DIR)
                       if f.lower().endswith((".jpg", ".jpeg", ".png"))])

# Only keep images that actually have a matching XML (skip e.g. 00018.jpg if it has none)
paired = []
for fname in image_files:
    stem = os.path.splitext(fname)[0]
    xml_path = os.path.join(ANNOTS_DIR, stem + ".xml")
    if os.path.exists(xml_path):
        paired.append((fname, xml_path))
    else:
        print(f"Skipping {fname}: no matching XML (no ground truth to train on)")

print(f"\nTotal usable image+XML pairs: {len(paired)}")

random.seed(42)
random.shuffle(paired)

n_val = max(1, int(len(paired) * VAL_RATIO))
val_pairs = paired[:n_val]
train_pairs = paired[n_val:]

print(f"Train: {len(train_pairs)} images | Val: {len(val_pairs)} images")

def build_split(pairs, img_out_dir, lbl_out_dir):
    for fname, xml_path in pairs:
        # copy image
        src_img = os.path.join(IMAGES_DIR, fname)
        dst_img = os.path.join(img_out_dir, fname)
        shutil.copy2(src_img, dst_img)

        # write YOLO label
        lines = voc_to_yolo_line(xml_path, CLASS_NAMES)
        stem = os.path.splitext(fname)[0]
        label_path = os.path.join(lbl_out_dir, stem + ".txt")
        with open(label_path, "w") as f:
            f.write("\n".join(lines))

build_split(train_pairs, YOLO_IMAGES_TRAIN, YOLO_LABELS_TRAIN)
build_split(val_pairs, YOLO_IMAGES_VAL, YOLO_LABELS_VAL)

print("\nYOLO dataset build complete.")
print("Train images:", len(os.listdir(YOLO_IMAGES_TRAIN)))
print("Val images:  ", len(os.listdir(YOLO_IMAGES_VAL)))



Total usable image+XML pairs: 50
Train: 40 images | Val: 10 images

YOLO dataset build complete.
Train images: 40
Val images:   10


## Cell 5 — Create data.yaml

In [ ]:

data_yaml_path = os.path.join(YOLO_DATASET_DIR, "data.yaml")

yaml_content = (
    f"path: {YOLO_DATASET_DIR}\n"
    "train: images/train\n"
    "val: images/val\n\n"
    "names:\n"
    "  0: UAV\n"
)

with open(data_yaml_path, "w") as f:
    f.write(yaml_content)

print("Saved:", data_yaml_path)
print(yaml_content)


Saved: C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone\yolo_dataset\data.yaml
path: C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone\yolo_dataset
train: images/train
val: images/val

names:
  0: UAV



## Cell 6 — Train YOLOv11 (fine-tune from pretrained weights)

In [ ]:

from ultralytics import YOLO

# yolo11n.pt = nano (fastest, least accurate)
# yolo11s.pt = small (good balance for a small dataset like this)
model = YOLO("yolo11s.pt")

results = model.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=960,       # keep close to your original image resolution to preserve small-object detail
    batch=8,
    patience=20,      # early stopping if no improvement
    project=os.path.join(BASE_DIR, "yolo_runs"),
    name="uav_detect",
    exist_ok=True,
    seed=42,
)


Ultralytics 8.4.92  Python-3.13.3 torch-2.13.0+cpu CPU (13th Gen Intel Core i7-13620H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone\yolo_dataset\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=uav_detect, nbs=64, nms=F

KeyboardInterrupt: 

## Cell 7 — Evaluate on validation set (Precision, Recall, mAP50, mAP50-95)

In [ ]:

best_weights = os.path.join(BASE_DIR, "yolo_runs", "uav_detect", "weights", "best.pt")
print("Using weights:", best_weights)

trained_model = YOLO(best_weights)
metrics = trained_model.val(data=data_yaml_path, imgsz=960)

print("\n===== YOLOv11 Evaluation Summary =====")
print(f"Precision:   {metrics.box.mp:.4f}")
print(f"Recall:      {metrics.box.mr:.4f}")
print(f"mAP50:       {metrics.box.map50:.4f}")
print(f"mAP50-95:    {metrics.box.map:.4f}")


Using weights: C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone\yolo_runs\uav_detect\weights\best.pt
Ultralytics 8.4.92  Python-3.13.3 torch-2.13.0+cpu CPU (13th Gen Intel Core i7-13620H)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 917.1376.9 MB/s, size: 117.3 KB)
val: Scanning C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone\yolo_dataset\labels\val.cache... 10 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 10/10 4.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.4s/it 2.4s
                   all         10         10      0.818      0.899      0.902      0.639
Speed: 1.9ms preprocess, 189.2ms inference, 0.0ms loss, 36.9ms postprocess per image
Results saved to C:\Users\HP\OneDrive\Documents\Desktop\vlm\scripts\runs\detect\val

===== YOLOv11 Evaluation Summary =====
Precision:   0.8180
Rec

## Cell 8 — Run inference on val images and visualize predictions

In [ ]:

import matplotlib.pyplot as plt

val_images = sorted(os.listdir(YOLO_IMAGES_VAL))[:6]  # preview first 6 val images

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, fname in enumerate(val_images):
    img_path = os.path.join(YOLO_IMAGES_VAL, fname)
    pred = trained_model.predict(img_path, imgsz=960, conf=0.25, verbose=False)[0]
    plotted = pred.plot()  # returns BGR numpy array with boxes drawn

    axes[i].imshow(plotted[..., ::-1])  # BGR -> RGB
    axes[i].set_title(fname)
    axes[i].axis("off")

plt.tight_layout()
plt.show()


<Figure size 1500x1000 with 6 Axes>

## Cell 9 — Save a summary report (for comparison against your VLM results)

In [ ]:

from datetime import datetime

report_path = os.path.join(BASE_DIR, "yolo11_drone_eval_summary.txt")

with open(report_path, "w") as f:
    f.write("YOLOv11 (fine-tuned) - UAV Detection Evaluation Report\n")
    f.write(f"Generated: {datetime.now().isoformat()}\n")
    f.write(f"Base weights: yolo11s.pt (fine-tuned)\n")
    f.write(f"Train images: {len(train_pairs)} | Val images: {len(val_pairs)}\n")
    f.write(f"Image size used: 960\n\n")
    f.write(f"Precision:   {metrics.box.mp:.4f}\n")
    f.write(f"Recall:      {metrics.box.mr:.4f}\n")
    f.write(f"mAP50:       {metrics.box.map50:.4f}\n")
    f.write(f"mAP50-95:    {metrics.box.map:.4f}\n\n")
    f.write("NOTE: Unlike the general VLMs tested (Qwen3-VL, LLaVA, Gemma3,\n")
    f.write("Qwen2.5-VL, Moondream) which scored near-zero mAP@0.5 out of the\n")
    f.write("box, YOLOv11 is a purpose-built detector fine-tuned directly on\n")
    f.write("this dataset's UAV annotations, so it is expected to perform\n")
    f.write("substantially better and serves as the practical baseline.\n")

print("Summary report saved to:", report_path)


Summary report saved to: C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone\yolo11_drone_eval_summary.txt
